## Load packages

In [ ]:


import json, os, random, time, shutil
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm


## Configuration  

In [ ]:
# --------------------------------------------------------------
# Configuration – adjust these paths
# --------------------------------------------------------------
INPUT_DIR = "data_pipeline/data_jsonl/"
OUTPUT_DIR = "data_pipeline/raft_book/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_DISTRACTORS = 4  # We'll have 1 relevant + 4 distractors
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

BATCH_SIZE = 16          # adjust based on GPU memory (try 16 if stable)
MAX_NEW_TOKENS = 100

In [ ]:
 # Find all .jsonl (and .json) files
all_files = list(Path(INPUT_DIR).rglob("*.json*"))   # catches both .json and .jsonl
print(f"Found {len(all_files)} files.")

Found 112 files.


## Load Qwen3 4B in 16‑bit  

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

##  Batched generation function

In [13]:
def batch_generate_qa(prompts, batch_size=BATCH_SIZE):
    responses = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating batches"):
        batch = prompts[i:i+batch_size]
        formatted = []
        for p in batch:
            messages = [{"role": "user", "content": p}]
            formatted.append(
                tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            )
        inputs = tokenizer(formatted, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
        with torch.inference_mode():
            outs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=0.2, do_sample=True,
                                  top_p=0.95, pad_token_id=tokenizer.eos_token_id)
        for j, out in enumerate(outs):
            gen = tokenizer.decode(out[inputs.input_ids[j].shape[0]:], skip_special_tokens=True).strip()
            responses.append(gen)
    return responses

 
## Main processing loop
 

In [ ]:
def process_file(file_path, output_dir):
    with open(file_path, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f if line.strip()]

    if len(chunks) < 5:
        return

    contents = [c["content"] for c in chunks]
    prompts = []
    meta = []   # list of (idx, chunk_data)

    for idx, cdata in enumerate(chunks):
        text = cdata["content"]
        if len(text) < 100: continue
        perspective = cdata.get("historian_perspective", "")
        historian = cdata.get("historian", "")
        domain = cdata.get("expert_domain", "")

        hint = ""
        if perspective and historian:
            hint = f"You are a {perspective} historian ({historian}). "
        elif perspective:
            hint = f"You are a {perspective} historian, who focus on {domain} period. "
        prompt = f"""{hint}Read the following text and generate a question whose answer is directly contained in the text. The question should reflect the historical perspective if mentioned. Then provide the exact answer.

            Text:
            \"\"\"
            {text}
            \"\"\"

            Reply with a JSON object containing exactly two keys: "question" and "answer".
            Only return the JSON, no extra text."""
        prompts.append(prompt)
        meta.append((idx, cdata))

    # Fast generation
    raw = batch_generate_qa(prompts)

    examples = []
    for i, r in enumerate(raw):
        r_clean = r.replace("```json", "").replace("```", "").strip()
        try:
            qa = json.loads(r_clean)
            q, a = qa.get("question",""), qa.get("answer","")
        except:
            continue
        if not q or not a: continue

        idx, orig = meta[i]
        relevant = contents[idx]
        other_idx = [j for j in range(len(contents)) if j != idx]
        random.shuffle(other_idx)
        distractors = []
        al = a.strip().lower()
        for j in other_idx:
            if al not in contents[j].lower():
                distractors.append(contents[j])
            if len(distractors) == 4:
                break
        if len(distractors) < 4: continue

        docs = [relevant] + distractors
        random.shuffle(docs)
        doc_list = [f"### Document [{k+1}]: {d}" for k,d in enumerate(docs)]
        ex = {
            "instruction": "You are a helpful assistant. Use the provided documents to answer the question. If the answer cannot be found in the documents, say 'I don't know'.",
            "documents": doc_list,
            "question": q,
            "output": a,
            "perspective": orig.get("historian_perspective",""),
            "historian": orig.get("historian",""),
            "expert_domain": orig.get("expert_domain", "")
        }
        examples.append(ex)

    out_name = file_path.stem + "_raft.jsonl"
    out_path = os.path.join(output_dir, out_name)
    with open(out_path, "w", encoding="utf-8") as fout:
        for e in examples:
            fout.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"✔️  {file_path.name} → {out_path} ({len(examples)} examples)")
    return out_path

## Main loop over all input files

In [ ]:
all_files = sorted(Path(INPUT_DIR).rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files.\n")

existing_outputs = set(os.listdir(OUTPUT_DIR)) if os.path.exists(OUTPUT_DIR) else set()


target_files = {
    "History_Of_The_Sikhs_Vol_"
}

selected_files = [f for f in all_files if f.stem.startswith(tuple(target_files))]
print(f"Selected {len(selected_files)} target files.\n")

processed_count = 0

for fpath in selected_files:
    out_name = fpath.stem + "_raft.jsonl"

    if out_name in existing_outputs:
        print(f"⏩ Skipping {fpath.name} (already processed)")
        continue

    processed_count += 1
    print(f"\n--- Processing file {processed_count}/{len(selected_files)}: {fpath.name} ---")
    process_file(fpath, OUTPUT_DIR)

print("\n🏁 Selected files processed. Outputs are in", OUTPUT_DIR)